# 🧠 GraphRAG — Colab Evaluation Notebook
Runs the full 30-question benchmark (LLM-Judge + BERTScore) on Google Colab.
Heavy GPU/RAM evaluation — designed to run independently from your local machine.

**Steps:**
1. Install dependencies
2. Set your API keys
3. Load FAISS index from repo
4. Run all 3 pipelines on 30 questions
5. Run BERTScore + LLM-Judge evaluation
6. Save results to benchmark_report.md

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q groq sentence-transformers faiss-cpu evaluate bert-score transformers \
    pyTigerGraph python-dotenv huggingface-hub streamlit pandas numpy wikipedia

In [ ]:
# ── Step 2: Clone the repo (or mount Drive if already cloned) ─────────────────
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/graph-RAG.git'  # ← update this
REPO_DIR = '/content/graphrag_project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} /content/graph-RAG
    REPO_DIR = '/content/graph-RAG/graphrag_project'

import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# ── Step 3: Set API keys ──────────────────────────────────────────────────────
import os
os.environ['GROQ_API_KEY']          = 'gsk_YOUR_KEY_HERE'          # ← required
os.environ['HF_TOKEN']              = 'hf_YOUR_TOKEN_HERE'          # ← for HF judge
os.environ['TIGERGRAPH_HOST']       = ''                             # ← optional
os.environ['TIGERGRAPH_PASSWORD']   = ''                             # ← optional
os.environ['TIGERGRAPH_GRAPH_NAME'] = 'GraphRAGDemo'
print('Keys set.')

In [ ]:
# ── Step 4: Verify FAISS index loads from repo ────────────────────────────────
import pickle, faiss, os

FAISS_PATH  = 'data/faiss_index.bin'
CHUNKS_PATH = 'data/faiss_chunks.pkl'

if os.path.exists(FAISS_PATH):
    idx = faiss.read_index(FAISS_PATH)
    with open(CHUNKS_PATH,'rb') as f: chunks = pickle.load(f)
    print(f'FAISS index: {idx.ntotal} vectors, {len(chunks)} chunks')
else:
    print('faiss_index.bin not found — building from Wikipedia...')
    from inference.pipelines import get_basic_rag
    rag = get_basic_rag()
    print(f'Built FAISS index: {rag.index.ntotal} vectors')

In [ ]:
# ── Step 5: Run all 3 pipelines on 30 ground-truth questions ──────────────────
import time
from inference.pipelines import run_pipeline_1_llm_only, run_pipeline_2_basic_rag, run_pipeline_3_graphrag
from eval.benchmark import GROUND_TRUTH

p1_answers, p2_answers, p3_answers = [], [], []
p1_tokens,  p2_tokens,  p3_tokens  = [], [], []
p1_times,   p2_times,   p3_times   = [], [], []

for i, gt in enumerate(GROUND_TRUTH, 1):
    q = gt['question']
    print(f'[{i:02d}/30] {q[:60]}')
    r1 = run_pipeline_1_llm_only(q)
    r2 = run_pipeline_2_basic_rag(q)
    r3 = run_pipeline_3_graphrag(q)
    p1_answers.append(r1.get('answer',''))
    p2_answers.append(r2.get('answer',''))
    p3_answers.append(r3.get('answer',''))
    p1_tokens.append(r1.get('total_tokens',0))
    p2_tokens.append(r2.get('total_tokens',0))
    p3_tokens.append(r3.get('total_tokens',0))
    p1_times.append(r1.get('response_time',0))
    p2_times.append(r2.get('response_time',0))
    p3_times.append(r3.get('response_time',0))
    time.sleep(0.4)  # rate limit

print('Done running all 30 questions.')

In [ ]:
# ── Step 6: Run LLM-Judge + BERTScore evaluation ──────────────────────────────
from eval.benchmark import evaluate_all_pipelines, generate_benchmark_report

hf_token = os.environ.get('HF_TOKEN', '')
acc = evaluate_all_pipelines(p1_answers, p2_answers, p3_answers, hf_token)

def avg(lst): return round(sum(lst)/len(lst), 2) if lst else 0

pipeline_metrics = {
    'p1_tokens': avg(p1_tokens), 'p2_tokens': avg(p2_tokens), 'p3_tokens': avg(p3_tokens),
    'p1_time':   avg(p1_times),  'p2_time':   avg(p2_times),  'p3_time':   avg(p3_times),
}

report_path = generate_benchmark_report(pipeline_metrics, acc)
print(f'Report saved to: {report_path}')

# Print summary
for name, r in acc.items():
    print(f"{name}: Judge={r['pass_percent']} | BERTScore={r['bertscore_f1']}")

In [ ]:
# ── Step 7: Display final report ──────────────────────────────────────────────
with open('eval/benchmark_report.md', 'r') as f:
    print(f.read())

In [ ]:
# ── Step 8 (optional): Download report to local machine ───────────────────────
try:
    from google.colab import files
    files.download('eval/benchmark_report.md')
    print('Downloaded benchmark_report.md')
except ImportError:
    print('Not running in Colab — report is at eval/benchmark_report.md')